# Analysis of Human Data 
This notebook loads the .dta file and runs the same preprocessing and descriptive analyses used for the LLM outputs, using the functions in `01_data_preparation/prep_functions.py` and `04_human_data_analysis/describe_functions.py`.

In [9]:
# Imports
import os
import sys
import pandas as pd
import pyreadstat
import importlib.util
from pathlib import Path
print('Python imports ready')

Python imports ready


In [10]:
# Utility: load local module by path (works for folders starting with digits)
def load_module_from_path(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

# Load prep_functions and describe_functions
repo_root = Path('..').resolve()
prep_path = repo_root / '01_data_preparation' / 'prep_functions.py'
desc_path = repo_root / '04_human_data_analysis' / 'describe_functions.py'
prep = load_module_from_path('prep_functions', str(prep_path))
desc = load_module_from_path('describe_functions', str(desc_path))
print('Loaded modules:', prep.__name__, desc.__name__)

Loaded modules: prep_functions describe_functions


In [11]:
# Read the .dta file (preserves labels/metadata)
dta_path = repo_root / 'VignettenAI_shareFolder' / 'VignettenAI_data03_long_format_2026-05-15.dta'
df, meta = pyreadstat.read_dta(str(dta_path))
print('Loaded .dta shape:', df.shape)
print('Column names:', list(df.columns))
try:
    print('Variable labels sample:', dict(list(meta.column_labels.items())[:10]))
except Exception:
    pass
df.head()

Loaded .dta shape: (19288, 78)
Column names: ['id', 'int_day', 'mobile', 'consent', 'age', 'ageGr', 'gender', 'genderalig', 'educ', 'educGr', 'countryofbirth', 'countryofbirth_other', 'race_a', 'race_b', 'race_c', 'race_d', 'race_e', 'race_f', 'race_g', 'race_h', 'ethnGr', 'maritalstat', 'attention', 'voted', 'votedcand', 'votepreference', 'votepreference_other', 'politicalleaning', 'religion', 'religion_other', 'religious', 'neighborpref_a', 'neighborpref_b', 'neighborpref_c', 'neighborpref_d', 'neighborpref_e', 'neighborpref_f', 'neighborpref_g', 'neighborpref_h', 'stereotype_a', 'stereotype_b', 'stereotype_c', 'stereotype_d', 'stereotype_e', 'stereotype_f', 'stereotype_g', 'stereotype_h', 'vignr', 'item', 'vignette', 'vig_race', 'vig_gender', 'vig_sex_orient', 'vig_religion', 'vigquest', 'comment', 'interviewtime', 'int_time', 'groupTime5367', 'groupTime3762', 'groupTime3774', 'groupTime3769', 'groupTime5365', 'groupTime3768', 'groupTime3775', 'groupTime3771', 'groupTime3770', 'grou

,id,int_day,mobile,consent,age,ageGr,gender,genderalig,educ,educGr,...,groupTime3773,groupTime3763,groupTime3778,groupTime3779,groupTime3780,groupTime5154,groupTime5155,groupTime5156,groupTime5157,groupTime3776
0,19,2025-12-12,0,1,67,5,2,2,3,2,...,36.03,14.19,9.63,18.84,21.89,8.8,9.51,31.32,11.6,7.2
1,19,2025-12-12,0,1,67,5,2,2,3,2,...,36.03,14.19,9.63,18.84,21.89,8.8,9.51,31.32,11.6,7.2
2,19,2025-12-12,0,1,67,5,2,2,3,2,...,36.03,14.19,9.63,18.84,21.89,8.8,9.51,31.32,11.6,7.2
3,19,2025-12-12,0,1,67,5,2,2,3,2,...,36.03,14.19,9.63,18.84,21.89,8.8,9.51,31.32,11.6,7.2
4,19,2025-12-12,0,1,67,5,2,2,3,2,...,36.03,14.19,9.63,18.84,21.89,8.8,9.51,31.32,11.6,7.2


In [12]:
# Create demographic columns from vignette attributes when missing
# Map numeric codes to strings for `race` and `gender` per user specification
race_map = {0: 'not_mentioned', 1: 'Black', 2: 'White', 3: 'Asian'}
gender_map = {0: 'not_mentioned', 1: 'man', 2: 'woman', 3: 'nonbinary'}

if 'vig_race' in df.columns:
    df['race'] = pd.to_numeric(df['vig_race'], errors='coerce').map(lambda x: race_map.get(int(x)) if pd.notna(x) else 'not_mentioned')

if 'vig_gender' in df.columns:
    df['gender'] = pd.to_numeric(df['vig_gender'], errors='coerce').map(lambda x: gender_map.get(int(x)) if pd.notna(x) else 'not_mentioned')

# Map religion and gender alignment from vignette fields
religion_map = {0: 'not_mentioned', 1: 'christian', 2: 'muslim', 3: 'jewish'}
gender_alignment_map = {0: 'not_mentioned', 1: 'cis', 2: 'trans'}

if 'vig_religion' in df.columns:
    df['religion'] = pd.to_numeric(df['vig_religion'], errors='coerce').map(lambda x: religion_map.get(int(x)) if pd.notna(x) else 'not_mentioned')

# Prefer `vig_sex_orient` for gender alignment, fall back to `genderalig` if present
if 'vig_sex_orient' in df.columns:
    df['gender_alignment'] = pd.to_numeric(df['vig_sex_orient'], errors='coerce').map(lambda x: gender_alignment_map.get(int(x)) if pd.notna(x) else 'not_mentioned')
elif 'genderalig' in df.columns:
    df['gender_alignment'] = df['genderalig']

print('Demographic columns created or preserved:', [c for c in ['race','gender','religion','gender_alignment'] if c in df.columns])

Demographic columns created or preserved: ['race', 'gender', 'religion', 'gender_alignment']


In [13]:
# Apply concept mapping (from prep_functions) if item_id present
df_proc = df.copy()
# Fallback: create demographic columns from vignette attributes when respondent-level columns missing
if 'race' not in df_proc.columns and 'vig_race' in df_proc.columns:
    df_proc['race'] = df_proc['vig_race'].apply(lambda x: f'vig_race_{int(x)}' if pd.notna(x) else x)
if 'gender' not in df_proc.columns and 'vig_gender' in df_proc.columns:
    df_proc['gender'] = df_proc['vig_gender'].apply(lambda x: f'vig_gender_{int(x)}' if pd.notna(x) else x)
if 'religion' not in df_proc.columns and 'vig_religion' in df_proc.columns:
    df_proc['religion'] = df_proc['vig_religion'].apply(lambda x: f'vig_religion_{int(x)}' if pd.notna(x) else x)
# Ensure response uses `vigquest` (human dataset response variable)
if 'vigquest' in df_proc.columns:
    df_proc['response'] = pd.to_numeric(df_proc['vigquest'], errors='coerce')
    print("Set 'response' from 'vigquest' (numeric)")

# Create `item_id`/`vignette_id` and `concept` where possible (prep will handle numeric `item`/`vignr`)
df_proc = prep.add_concept_mapping(df_proc)
print('Concepts added (or created where possible)')

# Recode demographics (adds cleaned category columns)
# Ensure gender_alignment exists (some .dta use 'genderalig')
if 'gender_alignment' not in df_proc.columns:
    if 'genderalig' in df_proc.columns:
        df_proc['gender_alignment'] = df_proc['genderalig']
    else:
        df_proc['gender_alignment'] = pd.NA
df_proc = prep.recode_demographic_columns(df_proc, df_name='VignettenAI')

# Clean response column (preserves original text and creates response_clean)
# Fallback: use 'vigquest' as the response column when present
if 'response' not in df_proc.columns and 'vigquest' in df_proc.columns:
    df_proc['response'] = df_proc['vigquest']
if 'response' in df_proc.columns:
    df_proc = prep.clean_response_column(df_proc, df_name='VignettenAI')
    print('Responses cleaned; statuses:', df_proc['response_status'].value_counts().to_dict())
else:
    print('No `response` column found — please inspect data and rename appropriate column to `response`.')

df_proc.head()

Set 'response' from 'vigquest' (numeric)
Applied simple item(1-4) -> concept mapping
Concepts added (or created where possible)

 Recoding demographic columns for VignettenAI...
   Race categories (not_mentioned=reference): ['not_mentioned', 'White', 'Black', 'Asian']
   Religion categories (not_mentioned=reference): ['not_mentioned', 'christian', 'muslim', 'jewish']
   Gender categories (not_mentioned=reference): ['not_mentioned', 'man', 'woman', 'nonbinary']
   Gender alignment categories (not_mentioned=reference): ['not_mentioned', 'cis', 'trans']
   Alternative scale columns: ['id', 'consent', 'age', 'ageGr', 'educ', 'educGr', 'countryofbirth', 'ethnGr', 'attention', 'voted', 'politicalleaning', 'vignr', 'item', 'vignette', 'vig_race', 'vig_gender', 'vig_sex_orient', 'vig_religion', 'interviewtime', 'int_time', 'groupTime5367', 'groupTime3762', 'groupTime3774', 'groupTime3769', 'groupTime5365', 'groupTime3768', 'groupTime3775', 'groupTime3771', 'groupTime3770', 'groupTime3772', 'gr

,id,int_day,mobile,consent,age,ageGr,gender,genderalig,educ,educGr,...,religion_cat,gender_cat,gender_alignment_cat,race_clean,gender_clean,religion_clean,gender_alignment_clean,response_text_original,response_status,response_clean
0,19,2025-12-12,0,1,67,5,man,2,3,2,...,not_mentioned,man,trans,not_mentioned,man,not_mentioned,trans,2.0,valid_numeric,2.0
1,19,2025-12-12,0,1,67,5,not_mentioned,2,3,2,...,not_mentioned,not_mentioned,trans,Asian,not_mentioned,not_mentioned,trans,-2.0,valid_numeric,-2.0
2,19,2025-12-12,0,1,67,5,not_mentioned,2,3,2,...,not_mentioned,not_mentioned,cis,White,not_mentioned,not_mentioned,cis,0.0,valid_numeric,0.0
3,19,2025-12-12,0,1,67,5,nonbinary,2,3,2,...,not_mentioned,nonbinary,not_mentioned,Black,nonbinary,not_mentioned,not_mentioned,-5.0,valid_numeric,-5.0
4,19,2025-12-12,0,1,67,5,not_mentioned,2,3,2,...,christian,not_mentioned,cis,not_mentioned,not_mentioned,christian,cis,-5.0,valid_numeric,-5.0


In [14]:
# Recode responses (reverse-poled concepts) using describe_functions
if 'concept' in df_proc.columns and 'response' in df_proc.columns:
    df_recoded = desc.recode_response_variable(df_proc, response_col='response', concept_col='concept', verbose=True)
else:
    df_recoded = df_proc.copy()
    print('Skipping recoding; required columns missing')

df_recoded.head()

=== RESPONSE VARIABLE RECODING ===
Original df shape: (19288, 95)
Response column: response
Concept column: concept
Concepts to be reverse-poled: ['AH', 'PH', 'ENV', 'CON', 'PIT']

Concept distribution:
concept
CON    4822
ADM    4822
PIT    4822
ENV    4822
Name: count, dtype: int64

Before recoding - Mean: 0.521, Std: 3.020
Rows to reverse: 14,466 out of 19,257 (75.1%)
After recoding  - Mean: 0.280, Std: 3.052

Mean response by concept after recoding:
  ADM: 1.603
  CON: -0.389 (REVERSED)
  ENV: 0.554 (REVERSED)
  PIT: -0.646 (REVERSED)

=== VERIFICATION ===
Comparing original vs recoded responses by concept group:

Reverse-poled concepts:
  Original mean: 0.160
  Recoded mean:  -0.160
  N = 14466

Normal concepts:
  Original mean: 1.603
  Recoded mean:  1.603
  N = 4822

=== RECODING COMPLETE ===
The following concepts have been reverse-poled (multiplied by -1):
  AH: 0 observations
  PH: 0 observations
  ENV: 4,822 observations
  CON: 4,822 observations
  PIT: 4,822 observations

N

,id,int_day,mobile,consent,age,ageGr,gender,genderalig,educ,educGr,...,gender_cat,gender_alignment_cat,race_clean,gender_clean,religion_clean,gender_alignment_clean,response_text_original,response_status,response_clean,response_recoded
0,19,2025-12-12,0,1,67,5,man,2,3,2,...,man,trans,not_mentioned,man,not_mentioned,trans,2.0,valid_numeric,2.0,-2.0
1,19,2025-12-12,0,1,67,5,not_mentioned,2,3,2,...,not_mentioned,trans,Asian,not_mentioned,not_mentioned,trans,-2.0,valid_numeric,-2.0,-2.0
2,19,2025-12-12,0,1,67,5,not_mentioned,2,3,2,...,not_mentioned,cis,White,not_mentioned,not_mentioned,cis,0.0,valid_numeric,0.0,-0.0
3,19,2025-12-12,0,1,67,5,nonbinary,2,3,2,...,nonbinary,not_mentioned,Black,nonbinary,not_mentioned,not_mentioned,-5.0,valid_numeric,-5.0,5.0
4,19,2025-12-12,0,1,67,5,not_mentioned,2,3,2,...,not_mentioned,cis,not_mentioned,not_mentioned,christian,cis,-5.0,valid_numeric,-5.0,5.0


In [15]:
# Collapse to vignette-level means (if item_id + vignette_id exist)
# Create aliases for item_id/vignette_id when dataset uses different column names
if 'item_id' not in df_recoded.columns and 'item' in df_recoded.columns:
    df_recoded['item_id'] = df_recoded['item']

In [16]:
# Compute per-row Warmth and Competence scores and save full processed with SCM
resp_col = 'response_recoded' if 'response_recoded' in df_recoded.columns else 'response'
warmth_polarity = {'ADM':1,'PIT':1,'ENV':-1,'CON':-1,'AF':1,'AH':-1,'PF':0,'PH':0}
competence_polarity = {'ADM':1,'PIT':-1,'ENV':1,'CON':-1,'AF':0,'AH':0,'PF':1,'PH':-1}
df_scm = df_recoded.copy()
df_scm['warmth_polarity'] = df_scm['concept'].map(warmth_polarity)
df_scm['competence_polarity'] = df_scm['concept'].map(competence_polarity)
df_scm['_scm_response'] = pd.to_numeric(df_scm[resp_col], errors='coerce')
df_scm['warmth_score'] = df_scm['warmth_polarity'] * df_scm['_scm_response']
df_scm['competence_score'] = df_scm['competence_polarity'] * df_scm['_scm_response']
df_scm['measures_warmth'] = df_scm['warmth_polarity'].fillna(0) != 0
df_scm['measures_competence'] = df_scm['competence_polarity'].fillna(0) != 0
df_scm.loc[~df_scm['measures_warmth'],'warmth_score'] = pd.NA
df_scm.loc[~df_scm['measures_competence'],'competence_score'] = pd.NA
out_full_scm = repo_root / '04_human_data_analysis' / 'vignettenai_full_processed_with_scm.csv'
df_scm.to_csv(out_full_scm, index=False)
print('Saved per-row SCM to:', out_full_scm)
df_scm.head()

Saved per-row SCM to: C:\Users\ThinkPad\Desktop\intersectional_bias_measure_fse\04_human_data_analysis\vignettenai_full_processed_with_scm.csv


,id,int_day,mobile,consent,age,ageGr,gender,genderalig,educ,educGr,...,response_status,response_clean,response_recoded,warmth_polarity,competence_polarity,_scm_response,warmth_score,competence_score,measures_warmth,measures_competence
0,19,2025-12-12,0,1,67,5,man,2,3,2,...,valid_numeric,2.0,-2.0,-1,-1,-2.0,2.0,2.0,True,True
1,19,2025-12-12,0,1,67,5,not_mentioned,2,3,2,...,valid_numeric,-2.0,-2.0,1,1,-2.0,-2.0,-2.0,True,True
2,19,2025-12-12,0,1,67,5,not_mentioned,2,3,2,...,valid_numeric,0.0,-0.0,1,-1,-0.0,-0.0,0.0,True,True
3,19,2025-12-12,0,1,67,5,nonbinary,2,3,2,...,valid_numeric,-5.0,5.0,-1,1,5.0,-5.0,5.0,True,True
4,19,2025-12-12,0,1,67,5,not_mentioned,2,3,2,...,valid_numeric,-5.0,5.0,-1,1,5.0,-5.0,5.0,True,True


In [17]:
# Compute SCM vignette-level warmth/competence scores and save CSV
# Use the per-row SCM if available (df_scm), else fall back to df_recoded
df_for_group = df_scm if 'df_scm' in globals() else df_recoded
# Ensure vignette_id prefers 'vignette' column when present and report source
if 'vignette_id' not in df_for_group.columns:
    if 'vignette' in df_for_group.columns:
        df_for_group['vignette_id'] = df_for_group['vignette']
        vignette_group_source = 'vignette'
    elif 'vignr' in df_for_group.columns:
        df_for_group['vignette_id'] = df_for_group['vignr']
        vignette_group_source = 'vignr'
    else:
        vignette_group_source = None
else:
    vignette_group_source = 'vignette_id (existing)'
print('Vignette grouping will use:', vignette_group_source)
if 'vignette_id' in df_for_group.columns and 'concept' in df_for_group.columns:
    scm = desc.compute_scm_scores(df_for_group, response_col=('response_recoded' if 'response_recoded' in df_for_group.columns else 'response'), concept_col='concept', groupby_col='vignette_id', keep_cols=['race','gender','religion'])
    print('SCM vignette-level shape:', scm.shape)
    out_vign = repo_root / '04_human_data_analysis' / 'vignettenai_vignette_level_scm.csv'
    scm.to_csv(out_vign, index=False)
    print('Saved vignette-level SCM to:', out_vign)
    scm.head()
else:
    print('Cannot compute SCM scores — required columns missing')


Vignette grouping will use: vignette_id (existing)
SCM vignette-level shape: (8, 8)
Saved vignette-level SCM to: C:\Users\ThinkPad\Desktop\intersectional_bias_measure_fse\04_human_data_analysis\vignettenai_vignette_level_scm.csv


In [18]:
# Additionally aggregate SCM by vignette text (if present) and save CSV (43-level aggregation)
if 'vignette' in df_scm.columns:
    grp_vign_text = df_scm.groupby('vignette').agg(
        n_responses=('warmth_score','count'),
        mean_warmth=('warmth_score','mean'),
        mean_competence=('competence_score','mean')
    ).reset_index()
    out_vign_text = repo_root / '04_human_data_analysis' / 'vignettenai_vignette_text_level_scm.csv'
    grp_vign_text.to_csv(out_vign_text, index=False)
    print('Saved vignette-text-level SCM to:', out_vign_text)
    grp_vign_text.head()
else:
    print('Column `vignette` not present; skipping vignette-text aggregation')

Saved vignette-text-level SCM to: C:\Users\ThinkPad\Desktop\intersectional_bias_measure_fse\04_human_data_analysis\vignettenai_vignette_text_level_scm.csv


## Human Data Analyses with myfunctions.py
This section mirrors the MAIHDA, bootstrap OLS, and intersectional bias-space analyses from `data_analysis.ipynb` on the human `.dta` data.

In [19]:
# Load myfunctions.py and run the same style of analyses on the human dataset
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
    sys.stderr.reconfigure(encoding='utf-8')
except Exception:
    pass
mf = load_module_from_path('myfunctions', str(repo_root / '04_human_data_analysis' / 'myfunctions.py'))
print('Loaded helper module:', mf.__name__)

import myfunctions as mf
from importlib import reload
mf = reload(mf)

analysis_df = df_scm.copy()
if 'gender_alignment' in analysis_df.columns and 'transness' not in analysis_df.columns:
    analysis_df = analysis_df.rename(columns={'gender_alignment': 'transness'})
elif 'genderalig' in analysis_df.columns and 'transness' not in analysis_df.columns:
    analysis_df['transness'] = analysis_df['genderalig']

vignette_cols = ['race', 'gender', 'religion', 'transness']
for col in vignette_cols:
    if col in analysis_df.columns:
        levels = ['not_mentioned'] + [x for x in analysis_df[col].dropna().unique().tolist() if x != 'not_mentioned']
        analysis_df[col] = pd.Categorical(analysis_df[col].fillna('not_mentioned'), categories=levels, ordered=False)

analysis_df = analysis_df.dropna(subset=['warmth_score', 'competence_score', 'race', 'gender', 'religion', 'transness', 'concept', 'vignette_id']).reset_index(drop=True)

analysis_df['stratum'] = analysis_df[vignette_cols].astype(str).agg('_'.join, axis=1)
print('Created analysis stratum with', analysis_df['stratum'].nunique(), 'unique strata')

df_strata = analysis_df[['stratum'] + vignette_cols].drop_duplicates().copy()

# MAIHDA-style analyses for Warmth and Competence
res_warmth_human_maihda = mf.run_maihda_simple(
    analysis_df,
    response_var='warmth_score',
    fixed_effects=('race', 'gender', 'religion', 'transness', 'concept'),
    stratum_col='stratum',
    out_dir='maihda_human_warmth',
    reml=True,
    optimizer='powell',
    disp=False
)

res_comp_human_maihda = mf.run_maihda_simple(
    analysis_df,
    response_var='competence_score',
    fixed_effects=('race', 'gender', 'religion', 'transness', 'concept'),
    stratum_col='stratum',
    out_dir='maihda_human_competence',
    reml=True,
    optimizer='powell',
    disp=False
)

mf.summarize_maihda(res_warmth_human_maihda, df_strata, out_dir='maihda_human_warmth_summary')
mf.summarize_maihda(res_comp_human_maihda, df_strata, out_dir='maihda_human_competence_summary')

mf.plot_intersectional_bias_space(
    res_warmth_human_maihda,
    res_comp_human_maihda,
    model_name='Human data',
    auto_save=True,
    save_dir='human_intersectional_bias_space'
)

# Cluster bootstrap OLS for the same covariates, clustered by vignette_id
bootstrap_formula = "{y} ~ C(gender) + C(race) + C(religion) + C(transness) + C(concept)"
boot_warmth = mf.cluster_bootstrap_ols_minimal(
    formula=bootstrap_formula.format(y='warmth_score'),
    df=analysis_df,
    cluster_col='vignette_id',
    B=500,
    seed=42
)
boot_warmth.insert(0, 'term', boot_warmth.index)
boot_warmth.insert(0, 'llm', 'human_warmth')
boot_warmth['term_simple'] = boot_warmth['term'].apply(lambda s: mf.simplify_term(s, keep_prefix=False))
boot_warmth.to_csv(repo_root / '04_human_data_analysis' / 'bootstrap_ols_human_warmth.csv')

boot_comp = mf.cluster_bootstrap_ols_minimal(
    formula=bootstrap_formula.format(y='competence_score'),
    df=analysis_df,
    cluster_col='vignette_id',
    B=500,
    seed=42
)
boot_comp.insert(0, 'term', boot_comp.index)
boot_comp.insert(0, 'llm', 'human_competence')
boot_comp['term_simple'] = boot_comp['term'].apply(lambda s: mf.simplify_term(s, keep_prefix=False))
boot_comp.to_csv(repo_root / '04_human_data_analysis' / 'bootstrap_ols_human_competence.csv')

boot_all_human = pd.concat([boot_warmth.reset_index(drop=True), boot_comp.reset_index(drop=True)], ignore_index=True)
fig, ax = mf.plot_ols_coefs_forest(
    boot_all_human[['llm', 'term_simple', 'coef', 'ci_low', 'ci_high']],
    title='Human data bootstrap OLS coefficients',
    sort_by='mean_abs'
)
fig.savefig(repo_root / '04_human_data_analysis' / 'human_bootstrap_ols_forest.png', dpi=300, bbox_inches='tight')
print('Saved human-data MAIHDA and bootstrap analyses')

ModuleNotFoundError: No module named 'statsmodels'

# FSE MAIHDA with respondent and stratum random effects
This block fits the three-level factorial-survey model: respondent random intercepts at L2 and stratum variance components at L3.

In [ ]:
# FSE MAIHDA with respondent and stratum random effects
from importlib import reload
mf = reload(mf)
fse_maihda = mf.run_fse_respondent_stratum_maihda(
    df,
    out_fname=str(repo_root / '04_human_data_analysis' / 'fse_respondent_stratum_maihda.xlsx'),
    plot_dir=str(repo_root / '04_human_data_analysis' / 'fse_respondent_stratum_maihda'),
    response_var='vigquest',
    respondent_col='id',
    stratum_col='stratum_id',
    stratum_from=('vig_race', 'vig_gender', 'vig_sex_orient', 'vig_religion'),
    fixed_effects=('vig_race', 'vig_gender', 'vig_sex_orient', 'vig_religion', 'item'),
    reference_level=0,
    item_reference='parent-teacher night',
    reml=False,
    optimizer='lbfgs',
    maxiter=3000,
    save_plots=True,
    verbose=True,
)
fse_maihda['summary']


FSE MAIHDA WITH RESPONDENT AND STRATUM RANDOM EFFECTS
Observations: 19,257
Respondents: 2,411
Strata: 43

MODELL 1A: NULL-MODELL
  σ²_Stratum    (L3): 4.5612 | VPC: 50.00%
  σ²_Respondent (L2): 0.0000 | VPC: 0.00%
  σ²_Residual   (L1): 4.5612 | VPC: 50.00%

MODELL 1B: ADDITIVES HAUPTEFFEKT-MODELL


c:\Users\ThinkPad\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


  σ²_Stratum    (L3): 4.2600 | VPC: 50.00%
  σ²_Respondent (L2): 0.0000 | VPC: 0.00%
  σ²_Residual   (L1): 4.2600 | VPC: 50.00%
  PCV Stratum: 6.6%
  PCV Respondent: nan%


,model,n_obs,n_respondents,n_strata,formula,tau_sq_stratum,tau_sq_respondent,sigma_sq,aic,bic,llf
0,1A,19257,2411,43,vigquest ~ 1,4.561192,0.0,4.561192,97227.049384,97250.646274,-48610.524692
1,1B,19257,2411,43,vigquest ~ C(vig_race) + C(vig_gender) + C(vig...,4.260016,0.0,4.260016,95939.583782,96073.299491,-47952.791891


In [ ]:
# OLS on vignette variables only — use categorical dummies so `not_mentioned` is dropped with drop_first=True
import pandas as pd
import statsmodels.api as sm

p = '04_human_data_analysis/vignettenai_full_processed_with_scm.csv'
df = pd.read_csv(p)

v_cols_map = {
    'race': ('race', 'vig_race'),
    'gender': ('gender', 'vig_gender'),
    'religion': ('religion', 'vig_religion'),
    'gender_alignment': ('gender_alignment', 'genderalig', 'vig_sex_orient')
}
for out_col, sources in v_cols_map.items():
    found = next((s for s in sources if s in df.columns), None)
    if found is None:
        df[out_col] = 'not_mentioned'
    else:
        if pd.api.types.is_integer_dtype(df[found]) or pd.api.types.is_float_dtype(df[found]):
            df[out_col] = df[found].apply(lambda x: f"vig_{int(x)}" if pd.notna(x) else 'not_mentioned')
        else:
            df[out_col] = df[found].fillna('not_mentioned').astype(str)
    unique_levels = [x for x in pd.unique(df[out_col].astype(str)) if x != 'not_mentioned']
    cats = ['not_mentioned'] + unique_levels
    df[out_col] = pd.Categorical(df[out_col].fillna('not_mentioned').astype(str), categories=cats, ordered=False)

predictor_cols = ['race', 'gender', 'religion', 'gender_alignment']

# Warmth
df_model_w = df.dropna(subset=['warmth_score'] + predictor_cols).copy()
# pass categorical columns directly so get_dummies respects category order
X_w = pd.get_dummies(df_model_w[predictor_cols], drop_first=True)
X_w = sm.add_constant(X_w).astype(float)
y_w = df_model_w['warmth_score'].astype(float)
res_w = sm.OLS(y_w, X_w).fit(cov_type='cluster', cov_kwds={'groups': df_model_w['id']})

# Competence
df_model_c = df.dropna(subset=['competence_score'] + predictor_cols).copy()
X_c = pd.get_dummies(df_model_c[predictor_cols], drop_first=True)
X_c = sm.add_constant(X_c).astype(float)
y_c = df_model_c['competence_score'].astype(float)
res_c = sm.OLS(y_c, X_c).fit(cov_type='cluster', cov_kwds={'groups': df_model_c['id']})

# Check that no dummy column includes 'not_mentioned'
print('Warmth design cols sample:', [c for c in X_w.columns if 'not_mentioned' in c])
print('Competence design cols sample:', [c for c in X_c.columns if 'not_mentioned' in c])

# Summarize and save
def summarize(res):
    out = res.params.to_frame('coef')
    out['se'] = res.bse
    out['t'] = res.tvalues
    out['p'] = res.pvalues
    ci = res.conf_int()
    out['ci_lower'] = ci[0]
    out['ci_upper'] = ci[1]
    return out

out_file = '04_human_data_analysis/human_ols_vignette_warmth_competence.xlsx'
with pd.ExcelWriter(out_file) as writer:
    summarize(res_w).to_excel(writer, sheet_name='warmth')
    summarize(res_c).to_excel(writer, sheet_name='competence')

print('Saved OLS summaries to', out_file)
print('\nWarmth model (coefficients, `not_mentioned` baseline):')
print(summarize(res_w).head(20))
print('\nCompetence model (coefficients, `not_mentioned` baseline):')
print(summarize(res_c).head(20))

In [1]:
# Cell: Display saved OLS vignette-only results
import pandas as pd
from IPython.display import display

out_file = '04_human_data_analysis/human_ols_vignette_warmth_competence.xlsx'
try:
    xl = pd.ExcelFile(out_file)
    df_w = pd.read_excel(xl, sheet_name='warmth', index_col=0)
    df_c = pd.read_excel(xl, sheet_name='competence', index_col=0)
    print('Warmth sheet shape:', df_w.shape)
    display(df_w)
    print('\nCompetence sheet shape:', df_c.shape)
    display(df_c)
except Exception as e:
    print('Could not read', out_file, '->', e)


Could not read 04_human_data_analysis/human_ols_vignette_warmth_competence.xlsx -> [Errno 2] No such file or directory: '04_human_data_analysis/human_ols_vignette_warmth_competence.xlsx'


In [ ]:
# Cell: Fit and print OLS vignette-only models (do not read Excel)
import pandas as pd
import statsmodels.api as sm
import numpy as np
from pathlib import Path

# Use repo_root when available to build an absolute path so reads succeed regardless of CWD
try:
    csv_path = str(repo_root / '04_human_data_analysis' / 'vignettenai_full_processed_with_scm.csv')
except Exception:
    csv_path = '04_human_data_analysis/vignettenai_full_processed_with_scm.csv'
if not Path(csv_path).exists():
    raise FileNotFoundError(f'Could not find {csv_path} - run the preprocessing cells that create df_scm first')
df = pd.read_csv(csv_path)

predictor_cols = ['race','gender','religion','gender_alignment']
# ensure categories with not_mentioned first
for col in predictor_cols:
    if col in df.columns:
        levels = [x for x in pd.unique(df[col].dropna().astype(str)) if x != 'not_mentioned']
        cats = ['not_mentioned'] + levels
        df[col] = pd.Categorical(df[col].fillna('not_mentioned').astype(str), categories=cats, ordered=False)

def fit_and_print(measure):
    df_model = df.dropna(subset=[measure] + predictor_cols).copy()
    X = pd.get_dummies(df_model[predictor_cols], drop_first=True)
    X = sm.add_constant(X).astype(float)
    y = df_model[measure].astype(float)
    try:
        res = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': df_model['id']})
    except Exception as e:
        print('Model fit error for', measure, '->', e)
        return None
    out = res.params.to_frame('coef')
    out['se'] = res.bse
    out['t'] = res.tvalues
    out['p'] = res.pvalues
    ci = res.conf_int()
    out['ci_lower'] = ci[0]
    out['ci_upper'] = ci[1]
    print('\n=====', measure, '=====', flush=True)
    print('Model rows:', df_model.shape[0])
    print('Design columns:', X.columns.tolist())
    print(out.to_string())
    return res

res_w = fit_and_print('warmth_score')
res_c = fit_and_print('competence_score')


FileNotFoundError: [Errno 2] No such file or directory: '04_human_data_analysis/vignettenai_full_processed_with_scm.csv'